# Biopython으로 인플루엔자 서열 분석해보기
## 개요
한타바이러스 하다가 생각난건데, NCBI에 최근에 유행했던 인플루엔자 데이터도 있는거 아닌가 해서 하게 됐습니다. 뭐 이것도 그렇게 복잡하지는 않아요. 근데 인플루엔자는 종류가 워낙 많다는게 특징... 

## 프로젝트 정보
- 인원: 1인(개인 프로젝트)
- 버전: 3.10(TF_base)
- 설치할 것들: Biopython, muscle
- 데이터 리소스: NCBI(Entrez로 갖고올 예정)

In [ ]:
# 모듈
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm # 넌 뭐냐 
import seaborn as sns

# BioPython
from Bio import Entrez, SeqIO # 왼쪽: 일단 털어보자/오른쪽: 시퀀스 다루려면 필요합니다. 필수임. 
from Bio import AlignIO # 서열 분석해줄 친구
from Bio import Phylo # 트리 그릴라면 필요해요 
from Bio.Phylo.TreeConstruction import DistanceCalculator, DistanceTreeConstructor
from Bio.Align import AlignInfo
from Bio.Align import MultipleSeqAlignment
from Bio.Phylo.BaseTree import BranchColor

# 통계분석용
from scipy.stats import mannwhitneyu
from itertools import combinations
from scipy.stats import spearmanr

import io # 누구세요?
import subprocess # 서브 프로세스(이건 또 뭐여...)
from collections import Counter
import re
import math

In [ ]:
# 그래프를 그리기 위한 기본 설정
plt.rcParams['font.family'] = 'Nanumsquare_ac' # 나눔바른펜(본인 기본 고딕 싫어함)
# plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['font.size'] = 14
plt.rcParams['axes.unicode_minus'] = False

# 사전세팅
Entrez.email = "blackholekun@gmail.com" # 이메일 
muscle_exe = "/opt/homebrew/bin/muscle" # 이거 경로 있어야 써요(which 치면 나옴)

# 데이터 가져오기
- 작년에 유행한 H3N2의 해마글루티닌 데이터를 가져옵니다. 

In [ ]:
# 인플루엔자 H3N2 서열 가져오기
# 쿼리 조건: 인플루엔자 A, 특정 아형, HA 유전자, 최근 1년(2025), 호스트가 사람 
query = f"Influenza A virus AND H3N2 AND HA[Gene Name] AND 2025[PDAT] AND Homo sapiens[Host]" # 사람독감 찾으려면... 

# 1. ID 리스트 가져오기
handle = Entrez.esearch(db="nucleotide", term=query, retmax=300)
record = Entrez.read(handle)
id_list = record["IdList"]
handle.close()

# 2. 실제 서열 데이터 가져오기 (FASTA 형식)
fetch_handle = Entrez.efetch(db="nucleotide", id=id_list, rettype="fasta", retmode="text")
sequences = list(SeqIO.parse(fetch_handle, "fasta"))
fetch_handle.close()

# 3. 저장 
with open("influenza_h3n2.fasta", "w") as f:
    SeqIO.write(sequences, f, "fasta")

print(f"성공적으로 {len(sequences)}개의 서열을 가져왔습니다.")
print("----------")

for record in sequences[:3]:
    print(f"ID: {record.id}")
    print(f"Description: {record.description}")
    print(f"Length: {len(record.seq)} bp\n")

## H3N2 필터링&정보 확인

In [ ]:
# 1. 먼저 H3N2 서열만 필터링해서 따로 모읍니다.
h3n2_only = [
    record for record in sequences 
    if "H3N2" in record.id or "H3N2" in record.description
]

# 2. 필터링된 서열들 중에서만 최소 길이를 찾습니다.
# (전체 sequences 기준이 아니라 h3n2_only 기준으로 해야 정확합니다)
min_len = min(len(s.seq) for s in h3n2_only)
print(f"H3N2 서열 개수: {len(h3n2_only)}")
print(f"맞춤 길이: {min_len} bp")

# 3. [중요] 필터링된 서열들을 자른 '새로운 리스트'를 만듭니다.
trimmed_h3n2 = []
for record in h3n2_only:
    trimmed_h3n2.append(record[:min_len])

# 4. [핵심] 이제 '자른 리스트'인 trimmed_h3n2를 넣어야 에러가 안 납니다!
alignment = MultipleSeqAlignment(trimmed_h3n2)

print(f"MultipleSeqAlignment 생성 성공! 서열 수: {len(alignment)}")

In [ ]:
# 사전통계-어느 지역 데이터를 얼마나 긁어왔는가? 
locations = []
for record in sequences:
    # Description에서 괄호 안의 지역 정보 추출 (예: A/Shanghai/...)
    match = re.search(r'A/([^/]+)/', record.description)
    if match:
        locations.append(match.group(1))

# 지역별 빈도수 확인
location_counts = Counter(locations)
print("--- 수집된 데이터 지역 분포 ---")
for loc, count in location_counts.most_common():
    print(f"{loc}: {count}개")

In [ ]:
# 지역별 라벨링(함수)
def clean_flu_labels(sequences):
    for record in sequences:
        # 1. 지역 추출 (A/지역/...)
        loc_match = re.search(r'A/([^/]+)/', record.description)
        location = loc_match.group(1) if loc_match else "Unknown"
        
        # 2. 연도 추출 (4자리 숫자)
        year_match = re.search(r'/(\d{4})', record.description)
        year = year_match.group(1) if year_match else "XXXX"
        
        # 3. 새로운 ID 생성 (예: 2023_Shanghai_H3N2)
        # 나중에 트리에 그릴 때 가독성을 위해 짧고 강렬하게!
        record.id = f"{year}_{location}"
        record.description = record.id # 설명도 통일
    return sequences

# 라벨 정리 실행
labeled_sequences = clean_flu_labels(sequences)

# 확인
for r in labeled_sequences[:5]:
    print(r.id)

# MSA

In [ ]:
print('MSA start... ')

# MSA 분석 시-작
try: 
    result = subprocess.run([muscle_exe, "-align", "influenza_h3n2.fasta", "-output", "influenza_h3n2_muscle_aligned.fasta"], check=True, capture_output=True, text=True)
    print("Completed. ")
except subprocess.CalledProcessError as e: 
    print(f"MSA failed: {e}")
finally:
    alignment = AlignIO.read("influenza_h3n2_muscle_aligned.fasta", "fasta")

# 오래 걸리니까 이거 돌려놓고 잠깐 바람 쐬고 오십쇼 

In [ ]:
print("====== MSA Result ======")
alignment = AlignIO.read("influenza_h3n2_muscle_aligned.fasta", "fasta") # FASTA 니네 확장자가 몇개냐... 

for record in alignment:
    print(f"{record.id[:10]:<15} : {record.seq[:100]}")

## 시각화

In [ ]:
# 1. Consensus 서열 계산 (전체 alignment 기준)
summary_align = AlignInfo.SummaryInfo(alignment)
# dumb_consensus는 가장 빈번한 염기를 선택합니다.
consensus = summary_align.dumb_consensus(threshold=0.5)

# 2. 분석 구간 설정 (559번 염기 주변 40bp)
# [주의] 만약 min_len이 580보다 작으면 에러가 날 수 있으니 체크가 필요합니다.
start, end = 540, 580

if end > alignment.get_alignment_length():
    print(f"경고: 현재 데이터 길이가 {alignment.get_alignment_length()}bp로 설정한 범위보다 짧습니다.")
    end = alignment.get_alignment_length()

subset_align = alignment[:, start:end]
# 인플루엔자 ID가 복잡할 수 있으니 간단하게 처리
names = [rec.id.split('.')[0] for rec in subset_align]

# 3. 데이터 수치화 (Consensus와 비교)
data = []
for record in subset_align:
    # subset_align의 인덱스는 0부터 시작하므로 consensus 접근 시 start를 더해줌
    row = [1 if record.seq[i] != consensus[start+i] else 0 for i in range(len(record.seq))]
    data.append(row)

# 4. 시각화
fig, ax = plt.subplots(figsize=(15, len(subset_align) * 0.3))
# pcolormesh가 더 선명하지만, 좁은 구간은 imshow도 깔끔합니다. 
# 대신 edgecolors를 주면 칸 구분이 잘 됩니다.
im = ax.imshow(data, aspect='auto', cmap='BuPu', interpolation='nearest')

# 축 설정
ax.set_yticks(range(len(names)))
ax.set_yticklabels(names, fontsize=8)
ax.set_xticks(range(0, end-start, 5))
ax.set_xticklabels(range(start, end, 5))
ax.set_title(f"2025 H3N2 HA 변이 핫스팟 ({start}-{end}bp)", fontsize=16, fontweight='bold', pad=25)
ax.set_xlabel("Nucleotide Position", fontsize=12, labelpad=20)

# 559번 위치 표시 (이번엔 그래프 안쪽 상단에 배치!)
target_pos = 559
target_idx = target_pos - start

if 0 <= target_idx < (end-start):
    ax.axvline(x=target_idx, color='red', linestyle='--', linewidth=2, alpha=0.5)
    
    # [방법 1] 그래프 내부 상단에 텍스트 배치 (y=-0.5는 첫 번째 서열 바로 위)
    ax.text(target_idx, -0.8, f'Pos {target_pos}', 
            color='blue', ha='center', va='bottom', fontweight='bold', fontsize=11,
            bbox=dict(facecolor='white', alpha=0.7, edgecolor='none', pad=1)) # 배경을 살짝 하얗게 해서 가독성 높임

# 그래프 전체 여백 조정 (아래쪽 여백 확보)
plt.subplots_adjust(bottom=0.2)

plt.tight_layout()
plt.show()

### POS 559

In [ ]:
# --- 설정값 ---
TOP_N = 30  
REAL_TARGET_POS = 559  # 엔트로피 피크 지점
start, end = REAL_TARGET_POS - 20, REAL_TARGET_POS + 20

# 1. 데이터 슬라이싱
subset_align = alignment[:TOP_N, start:end]
names = [rec.id.split('.')[0] for rec in subset_align] # ID 포맷 대응

# 2. 데이터 수치화
data = []
for record in subset_align:
    # subset_align의 인덱스 i는 0부터 시작하므로 consensus 비교 시 start+i를 사용합니다.
    row = [1 if record.seq[i] != consensus[start+i] else 0 for i in range(len(record.seq))]
    data.append(row)

# 3. 시각화 
fig, ax = plt.subplots(figsize=(15, 10), dpi=200) # 시퀀스 30개이므로 높이를 10으로 확보
im = ax.imshow(data, aspect='auto', cmap='BuPu', interpolation='nearest')

# 축 설정
ax.set_yticks(range(len(names)))
ax.set_yticklabels(names, fontsize=10)
ax.set_xticks(range(0, end-start, 5))
ax.set_xticklabels(range(start, end, 5), fontsize=10)

# 제목 및 라벨 (겹침 방지를 위해 pad와 labelpad 조절)
ax.set_title(f"2025 H3N2 HA 변이 집중 분석: Pos {REAL_TARGET_POS} 주변", fontsize=18, fontweight='bold', pad=35)
ax.set_xlabel("Nucleotide Position (bp)", fontsize=13, labelpad=15)

# --- 타겟 라인 및 라벨 표시 ---
# target_idx는 0부터 시작하는 '상대적 인덱스'입니다.
target_idx = REAL_TARGET_POS - start

if 0 <= target_idx < (end-start):
    ax.axvline(x=target_idx, color='red', linestyle='--', alpha=0.6, linewidth=2.5)
    
    # [수정] 라벨을 X축 숫자와 안 겹치게 '그래프 내부 상단'으로 배치
    # y = -1.0은 첫 번째 서열 바로 위 공간입니다.
    ax.text(target_idx, -1.2, f'Target: Pos {REAL_TARGET_POS}', color='blue', 
            ha='center', va='bottom', fontweight='bold', fontsize=12,
            bbox=dict(facecolor='white', alpha=0.8, edgecolor='none', pad=2))

plt.tight_layout()
plt.savefig("H3N2_Mutation_Focus_1734.png", dpi=300, bbox_inches='tight')
plt.show()

### POS 1734

In [ ]:
# --- 설정값 ---
TOP_N = 30  
REAL_TARGET_POS = 1734  # 엔트로피 피크 지점
start, end = REAL_TARGET_POS - 20, REAL_TARGET_POS + 20

# 1. 데이터 슬라이싱
subset_align = alignment[:TOP_N, start:end]
names = [rec.id.split('.')[0] for rec in subset_align] # ID 포맷 대응

# 2. 데이터 수치화
data = []
for record in subset_align:
    # subset_align의 인덱스 i는 0부터 시작하므로 consensus 비교 시 start+i를 사용합니다.
    row = [1 if record.seq[i] != consensus[start+i] else 0 for i in range(len(record.seq))]
    data.append(row)

# 3. 시각화 
fig, ax = plt.subplots(figsize=(15, 10), dpi=200) # 시퀀스 30개이므로 높이를 10으로 확보
im = ax.imshow(data, aspect='auto', cmap='BuPu', interpolation='nearest')

# 축 설정
ax.set_yticks(range(len(names)))
ax.set_yticklabels(names, fontsize=10)
ax.set_xticks(range(0, end-start, 5))
ax.set_xticklabels(range(start, end, 5), fontsize=10)

# 제목 및 라벨 (겹침 방지를 위해 pad와 labelpad 조절)
ax.set_title(f"2025 H3N2 HA 변이 집중 분석: Pos {REAL_TARGET_POS} 주변", fontsize=18, fontweight='bold', pad=35)
ax.set_xlabel("Nucleotide Position (bp)", fontsize=13, labelpad=15)

# --- 타겟 라인 및 라벨 표시 ---
# target_idx는 0부터 시작하는 '상대적 인덱스'입니다.
target_idx = REAL_TARGET_POS - start

if 0 <= target_idx < (end-start):
    ax.axvline(x=target_idx, color='red', linestyle='--', alpha=0.6, linewidth=2.5)
    
    # [수정] 라벨을 X축 숫자와 안 겹치게 '그래프 내부 상단'으로 배치
    # y = -1.0은 첫 번째 서열 바로 위 공간입니다.
    ax.text(target_idx, -1.2, f'Target: Pos {REAL_TARGET_POS}', color='blue', 
            ha='center', va='bottom', fontweight='bold', fontsize=12,
            bbox=dict(facecolor='white', alpha=0.8, edgecolor='none', pad=2))

plt.tight_layout()
plt.savefig("H3N2_Mutation_Focus_1734.png", dpi=300, bbox_inches='tight')
plt.show()

- POS 559: point mutation 발생
- POS 1734: 대국적 변이 발생

# 섀넌 엔트로피 분석

In [ ]:
# 섀넌 엔트로피 점수 도출
def get_top_variable_sites_no_gap(alignment, top_n=10):
    length = alignment.get_alignment_length()
    variability = []

    ref_seq = alignment[0].seq

    for i in range(length):
        # 🔴 reference가 gap이면 무조건 스킵
        if ref_seq[i] == '-':
            continue

        column = alignment[:, i].replace("-", "")
        if not column:
            continue

        counts = Counter(column)
        total = sum(counts.values())

        entropy = 0.0
        for c in counts.values():
            p = c / total
            entropy -= p * math.log2(p)

        variability.append((i, entropy))

    return sorted(variability, key=lambda x: x[1], reverse=True)[:top_n]

def alignment_to_sequence_pos(aligned_seq, aln_pos):
    count = 0
    for i in range(aln_pos + 1):
        if aligned_seq[i] != '-':
            count += 1
    return count


ref_seq = alignment[0].seq
top_sites = get_top_variable_sites_no_gap(alignment, top_n=10)

high_entropy_ha_sites = []

print("--- 변이가 집중된 주요 포지션 분석 결과 ---")
for aln_pos, score in top_sites:
    real_pos = alignment_to_sequence_pos(ref_seq, aln_pos)
    high_entropy_ha_sites.append(real_pos)
    print(f"Alignment {aln_pos:4d} → HA Pos {real_pos:4d} | 엔트로피: {score:.3f}")

print("\n최종 고엔트로피 HA 포지션 리스트:")
print(high_entropy_ha_sites)

In [ ]:
def shannon_entropy_no_gap(alignment):
    length = alignment.get_alignment_length()
    ref_seq = alignment[0].seq

    entropy_scores = []

    for i in range(length):
        # reference가 gap이면 제외
        if ref_seq[i] == '-':
            entropy_scores.append(np.nan)
            continue

        column = alignment[:, i].replace("-", "")
        if not column:
            entropy_scores.append(np.nan)
            continue

        counts = Counter(column)
        total = sum(counts.values())

        entropy = 0.0
        for c in counts.values():
            p = c / total
            entropy -= p * math.log2(p)

        entropy_scores.append(entropy)

    return np.array(entropy_scores)

def sliding_window_mean(values, window=20):
    """
    values : np.array (entropy scores, np.nan 포함)
    window : window size
    """
    smoothed = []

    for i in range(len(values)):
        start = max(0, i - window // 2)
        end = min(len(values), i + window // 2 + 1)

        window_vals = values[start:end]
        window_vals = window_vals[~np.isnan(window_vals)]

        if len(window_vals) == 0:
            smoothed.append(np.nan)
        else:
            smoothed.append(np.mean(window_vals))

    return np.array(smoothed)

In [ ]:
entropy_raw = shannon_entropy_no_gap(alignment)
entropy_window = sliding_window_mean(entropy_raw, window=25)

In [ ]:
plt.figure(figsize=(15, 5))

plt.plot(entropy_window, color='blue', linewidth=2)
plt.fill_between(
    range(len(entropy_window)),
    entropy_window,
    color='blue',
    alpha=0.25
)

target_pos = 559
plt.axvline(target_pos, color='red', linestyle='--', alpha=0.6)
plt.text(
    target_pos, 
    max(entropy_window)*0.9,
    "HA hotspot (559)",
    color='red',
    ha='center',
    fontweight='bold'
)

plt.title("Influenza HA Sliding Window Shannon Entropy (gap excluded)")
plt.xlabel("Alignment Position")
plt.ylabel("Mean Shannon Entropy")
plt.tight_layout()
plt.show()

### 통계분석 (섀넌 엔트로피)
- 귀무가설: 작년에 유행한 인플루엔자 H3N2의 해마글루티닌 변이는 무작위적으로 발생하며, 특정 위치에 선호적으로 집중되지 않는다.
- 대립가설: 작년에 유행한 인플루엔자 H3N2의 해마글루티닌 변이는 무작위가 아니며, 특정 위치(hotspots)에 유의하게 집중된다.

In [ ]:
# 여러분 이것도 통계분석이 됩니다. 
variation_scores = np.array(entropy_list)

mean_var = np.mean(variation_scores)
median_var = np.median(variation_scores)
iqr_var = np.percentile(variation_scores, 75) - np.percentile(variation_scores, 25)

print(f"Mean variation score: {mean_var:.4f}")
print(f"Median variation score: {median_var:.4f}")
print(f"IQR: {iqr_var:.4f}")

In [ ]:
# Define high-variation hotspots (top 10%)
threshold = np.percentile(variation_scores, 90)

hotspots = variation_scores[variation_scores >= threshold]
non_hotspots = variation_scores[variation_scores < threshold]

u_stat, p_value = mannwhitneyu(
    hotspots,
    non_hotspots,
    alternative="greater"
)

print(f"Hotspot threshold (90th percentile): {threshold:.4f}")
print(f"Mann–Whitney U statistic: {u_stat:.1f}")
print(f"p-value: {p_value:.4e}") # 아 이거는 제가 소수점 조절을 못했어요... 하면 큰일나... 

- P-value < 0.001이므로 작년에 유행한 인플루엔자 H3N2의 해마글루티닌 변이는 무작위적으로 발생하며, 특정 위치에 선호적으로 집중되지 않는다는 귀무가설을 기각함. 
> 작년에 유행한 인플루엔자 H3N2의 해마글루티닌 변이는 무작위가 아니며, 특정 위치(hotspots)에 유의하게 집중된다.

# Phylogenic tree

In [ ]:
# 트! 리! 
calculator = DistanceCalculator('identity')
dm = calculator.get_distance(alignment)
constructor = DistanceTreeConstructor(calculator, 'nj')
tree = constructor.build_tree(alignment)

terms = tree.get_terminals()
x_limit = max([tree.distance(t) for t in terms])
fig = plt.figure(figsize=(20, 60), dpi=150) # 난 해상도 설정도 될 줄 몰랐고... 
ax = fig.add_subplot(1, 1, 1)

for clade in tree.get_terminals():
    original_name = str(clade.name)
    if '_' in original_name:
        parts = original_name.split('_')
        clade.name = f"[{parts[0]}] {parts[1]} ({original_name})"
    else:
        clade.name = original_name

Phylo.draw(tree, axes=ax, do_show=False, label_func=lambda x: "", show_confidence=False)

# 내가 진짜 이것때문에 제미나이랑 급나 씨름했는데 색깔이 안바껴요. 
# 이름도 몇번이나 했는데 ID만 줄창떠요. 아오. 
for i, node in enumerate(terms):
    y_pos = i + 1  # 가지의 y축 위치
    x_pos = tree.distance(node) # 가지가 끝나는 x축 위치
    
    orig_name = str(node.name)
    # 이름 가공: [연도] 지역 (ID)
    if '_' in orig_name:
        p = orig_name.split('_')
        # 혹시 이미 가공된 이름이라면 중복 방지
        display_text = f"  ◀ [{p[0]}] {p[1]}" if '[' not in orig_name else f"  ◀ {orig_name}"
    else:
        display_text = f"  ◀ {orig_name}"
    
    # 가지 끝(x_pos)에 바로 텍스트를 박습니다.
    ax.text(x_pos, y_pos, display_text, 
            va='center', ha='left', 
            fontsize=14, 
            fontweight='bold' if "LC909067" in orig_name else 'normal')

ax.set_xlim(0, x_limit * 1.8) 
ax.set_ylim(0, len(terms) + 2)
ax.set_axis_off() # 축 숫자 빠잉 

plt.rc('font', size=14) # 내부 글꼴 사이즈
plt.rc('axes', titlesize=20) # 제모옥은 이 크기로 하겠습니다 
plt.title("Influenza A (H3N2) HA Phylogenetic Tree by Region/Year")
plt.tight_layout()
plt.savefig("Influenza_H3N2_Final_Tree.png", dpi=300, bbox_inches='tight')
plt.xlabel("Genetic Distance (Substitutions per site)")
plt.show()

## 통계분석
- 귀무가설: 트리 거리와 서열 유사도는 상관이 없다 → 계통수 구조는 서열 차이를 반영하지 않는다.
- 대립가설: 트리 거리와 서열 유사도간에 서로 상관이 있다 → 계통수 구조는 서열 차이를 반영했다. 

In [ ]:
tree_distances = []
seq_identities = []

terms = tree.get_terminals()

def pairwise_identity(seq1, seq2):
    # 두 서열 중 어느 한쪽이라도 갭이 아닌 위치만 골라냄
    matches = 0
    total_valid_length = 0
    for s1, s2 in zip(seq1, seq2):
        if s1 == '-' and s2 == '-': # 둘 다 갭이면 무시
            continue
        total_valid_length += 1
        if s1 == s2:
            matches += 1
    
    return (matches / total_valid_length) if total_valid_length > 0 else 0

for rec1, rec2 in combinations(alignment, 2):
    id1 = rec1.id.split('.')[0]
    id2 = rec2.id.split('.')[0]
    
    try:
        # 가공된 트리 이름 속에서 원본 ID가 포함된 노드를 각각 찾음
        node1 = [t for t in terms if id1 in t.name][0]
        node2 = [t for t in terms if id2 in t.name][0]
        
        d = tree.distance(node1, node2)
        iden = pairwise_identity(str(rec1.seq), str(rec2.seq))
        
        tree_distances.append(d)
        seq_identities.append(iden)
    except IndexError:
        # 트리에 해당 ID가 없는 경우 건너뜀
        continue

In [ ]:
rho, p = spearmanr(tree_distances, seq_identities)

print(f'Rho: {rho:.4f}')
print(f'P-value: {p:.4e}')

### 시각화

In [ ]:
plt.figure(figsize=(10, 7), dpi=120)
sns.regplot(x=tree_distances, y=seq_identities, 
            scatter_kws={'alpha':0.2, 'color':'gray', 's':10}, 
            line_kws={'color':'red', 'label': f'Spearman Rho: {rho:.3f}'})

plt.title("H3N2 HA: Tree Distance vs Sequence Identity", fontsize=15, pad=15)
plt.xlabel("Genetic Distance on Tree", fontsize=12)
plt.ylabel("Pairwise Sequence Identity", fontsize=12)
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

- 한타바이러스와 통계검정 방식이 다르죠? 이유는 간단합니다. 쟤는 그냥 한 아종에서 조금씩 변이돼서 갈라진거라 변이가 연속적입니다. 그래서 한타바이러스처럼 clade로 자르기가 애매해요. 
- 그래서 트리 거리와 서열 유사도의 상관관계를 스피어만 상관계수로 본 겁니다. 근데 왜 쟤가 음수냐고요? 트리간에 거리가 멀 수록 유사도도 떨어지니까요. 
- P < 0.001이므로 귀무가설은 기각해도 됩니다. 
> 트리 거리와 서열 유사도간에 서로 상관이 있다 → 계통수 구조는 서열 차이를 반영했다. 

- 저 멀리 떨어져 있는 애들의 정체가 이거라는 얘기입니다. 

# 변이도가 가장 높은 바이러스는? 

In [ ]:
# 사람을 공격하는 바이러스만 찾습니다 
human_terms = [t for t in tree.get_terminals() if 'canine' not in str(t.name).lower()]

# 유전적 거리순으로 정렬 
human_distances = [(tree.distance(t), t.name) for t in human_terms]
human_distances.sort(key=lambda x: x[0], reverse=True)

print("=== 🚨 독감 변종 TOP 5 ===")
print("-" * 70)
print(f"{'순위':<4} | {'ID':<12} | {'변이도':<8} | {'신상 정보'}")
print("-" * 70)

for i, (dist, name) in enumerate(human_distances[:5], 1):
    target_id = str(name)
    found_info = "정보 없음"
    
    # alignment 데이터에서 상세 지역/연도 정보 매칭
    for record in alignment:
        if target_id in record.description or target_id in record.id:
            full_info = record.description if record.description else record.id
            if '_' in full_info:
                parts = full_info.split('_')
                found_info = f"[{parts[0]}] {parts[1].split(' ')[0]}"
            else:
                # description에서 연도/지역 추출 시도 (괄호 안 정보 등)
                found_info = full_info.split('virus (')[1].split(')')[0] if '(' in full_info else full_info
            break
            
    print(f"{i:<5} | {target_id:<12} | {dist:.4f} | {found_info}")

print("-" * 70)
print("※ 변이도가 높을수록 기존 면역 체계를 회피할 가능성이 클 수도 있습니다.")